In [ ]:
# Base imports
import os
import pickle

# Compute imports
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm, trange

# Plotting imports
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
from matplotlib import pyplot as plt
import seaborn as sns
from plotly import express as px

# ML import
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error, median_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from pyphylon.util import load_config

In [ ]:
# Load in L_binarized matrix
L_binarized = pd.read_csv("/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/nmf-outputs/L_binarized.csv", index_col=0)
L_binarized

## Process Panaroo Results to Generate Representative Fasta File
use for eggnog later

In [ ]:
import networkx as nx

G = nx.read_gml('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/final_graph.gml')

In [ ]:
print("Writing Representative Sequences to File for each Gene...")
with open('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/representative_alleles.faa', 'w') as f:
    for node, attributes in tqdm(G.nodes.data(True)):
        name = attributes['name']

        prot_seqs = attributes['protein'].split(';')
        max_len = 0
        max_prot = ""

        for prot in prot_seqs:
            if "*" in prot:
                continue

            if len(prot) > max_len:
                max_len = len(prot)
                max_prot = prot

        if max_prot == "":
            print(prot_seqs)
            break
        prot_seq = max_prot

        f.write(">" + name + '\n')
        f.write(prot_seq+'\n')

## Generate Header to Allele Translations

In [ ]:
panaroo_df_genes = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/gene_presence_absence.csv', index_col='Gene', low_memory = False, dtype=object).drop(['Non-unique Gene name', 'Annotation'], axis=1)
gene_information = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/gene_data.csv', dtype=object)
genomes = list(panaroo_df_genes.columns)
graph = G

In [ ]:
gene_information = gene_information.set_index('clustering_id')
panaroo_dict = {}  # use a plain dict first
header_dict = {}
for node, attributes in tqdm(G.nodes.data(True)):
    name = attributes['name']
    seq_ids = attributes['seqIDs']
    for seq_id in seq_ids:
        panaroo_dict[seq_id] = name
        header_dict[gene_information.loc[seq_id, 'annotation_id']] = name

# Convert to pandas Series at the end
panaroo_header_to_allele = pd.Series(panaroo_dict)
header_to_allele = pd.Series(header_dict)

panaroo_header_to_allele.to_pickle('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/panaroo_header_to_allele.pickle.gz', compression="gzip")
header_to_allele.to_pickle('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/header_to_allele.pickle.gz', compression="gzip")

gene_information = gene_information.reset_index()

In [ ]:
num_members = {}
for node, attributes in tqdm(graph.nodes.data(True)):
    num_members[attributes['name']] = len(attributes['seqIDs'])

num_members = pd.Series(num_members)

## Save Paralog Information

In [ ]:
from collections import defaultdict
centroids_to_nodes = defaultdict(list)

In [ ]:
for node, attributes in graph.nodes.data(True):
    if attributes['paralog']:
        for centroid in attributes['centroid'].split(';'):
            centroids_to_nodes[centroid].append(attributes['name'])

In [ ]:
paralogous_groups_by_gene = defaultdict(set)

for key, item in centroids_to_nodes.items():
    for gene in item:
        paralogous_groups_by_gene[gene] = paralogous_groups_by_gene[gene].union(set(item))

In [ ]:
all_paralogs = set()
for genes in paralogous_groups_by_gene.values():
    all_paralogs = all_paralogs.union(set(genes))

In [ ]:
parent = {}

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    parent[find(x)] = find(y)
    
# use union find to create disjoint sets of paralogs
for genes in paralogous_groups_by_gene.values():
    for gene in genes:
        if gene not in parent:
            parent[gene] = gene

for genes in paralogous_groups_by_gene.values():
    genes = list(genes)
    for i in range(1, len(genes)):
        union(genes[0], genes[i])

# Collect groups
groups = defaultdict(set)
for elem in parent:
    groups[find(elem)].add(elem)

# Final unique groupings
unique_groups = list(groups.values())
print(len(unique_groups))

In [ ]:
import pickle

# make a dictionary for checking paralogs and save for future use
genes_by_paralog_group = {}

for group in unique_groups:
    for gene in group:
        genes_by_paralog_group[gene] = group

with open('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/gene_by_paralog_group.pickle', "wb") as f:
    pickle.dump(genes_by_paralog_group, f)

## Functional enrichment analysis

In [ ]:
# collection functions...

from pyphylon.biointerp import collect_functions
# # only run me once:
# all_functions = collect_functions('/mnt/craig/pan_phylon/P_aeruginosa/data/', 'processed_ncbi/bakta/')
# all_functions.to_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/all_functions.csv')

In [ ]:
all_functions = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/all_functions.csv', index_col=0)

In [ ]:
# Get the pan-genome
df_genes = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/gene_presence_absence.Rtab', sep = '\t').set_index('Gene')

In [ ]:
df = pd.DataFrame(pd.read_pickle('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/panaroo_results/header_to_allele.pickle.gz')).reset_index()
df.head()

In [ ]:
from pyphylon.biointerp import get_pg_to_locus_map
# Data wrangling to get the functions for each cluster            
pg2locus_map = get_pg_to_locus_map('/mnt/craig/pan_phylon/P_aeruginosa/data/', 'PAeruginosa')
pg2locus_map

In [ ]:
from pyphylon.biointerp import get_pg_to_locus_map
# Data wrangling to get the functions for each cluster            
pg2locus_map = get_pg_to_locus_map('/mnt/craig/pan_phylon/P_aeruginosa/data/', 'PAeruginosa')
functions2genes = pd.merge(all_functions, pg2locus_map, left_on='locus', right_on='gene_id')
cluster_functions = functions2genes.groupby('cluster').first().reset_index()[['cluster','product','go']]
cluster_functions

In [ ]:
from pyphylon.biointerp import explode_go_annos
cluster_to_go_functions = explode_go_annos(cluster_functions)
cluster_to_go_functions

In [ ]:
go_functions_count = cluster_to_go_functions.groupby('go').count()
go_functions = go_functions_count[go_functions_count['cluster'] > 3].sort_values('cluster', ascending=False)
go_functions

In [ ]:
#calculate a single engirchment
from pyphylon.biointerp import calc_enrichment
go_term = 'GO:0016032'
phylon = 'unchar_1'
calc_enrichment(L_binarized, cluster_to_go_functions, go_term, functions2genes, phylon, phylon_contribution_cutoff=0)  

In [ ]:
from pyphylon.biointerp import calc_all_phylon_go_enrichments, get_go_mapping  # TODO need to speed this up - shrinking functions2genes to only accessory genes seemed to help...

phylon_go_enrichments = calc_all_phylon_go_enrichments(L_binarized, functions2genes, cluster_to_go_functions, go_functions, phylon_contribution_cutoff=0.5)
phylon_go_enrichments = phylon_go_enrichments[phylon_go_enrichments['p_value']<0.05]

go_mapping = get_go_mapping()
phylon_go_enrichments = pd.merge(phylon_go_enrichments, go_mapping, left_on='function', right_index=True, how='left')
missing_go = phylon_go_enrichments[phylon_go_enrichments['name'].isnull()].index
phylon_go_enrichments.loc[missing_go, 'name'] = phylon_go_enrichments.loc[missing_go,'function']

phylon_go_enrichments = phylon_go_enrichments[phylon_go_enrichments['function']!='SO:0001217'] # filter out SO:0001217 is just a category for "protein encoding gene"
phylon_go_enrichments.to_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/phylon_go_enrichment.csv')

phylon_go_enrichments = pd.read_csv('/mnt/craig/pan_phylon/P_aeruginosa/data/processed_ncbi/phylon_go_enrichment.csv', index_col=0)

In [ ]:
phylon_go_enrichments.head()

In [ ]:
phylon_go_enrichments_mat = pd.pivot_table(phylon_go_enrichments, index='phylon', columns='function', values='p_value')
sns.clustermap(phylon_go_enrichments_mat.fillna(0.05), cmap='rocket_r')
plt.title('phylon functional enrichments')

In [ ]:
# Explore a single phylon:
phylon = 'ST253'
phylon_go_enrichments[phylon_go_enrichments['phylon']==phylon]

In [ ]:
from statsmodels.stats.multitest import fdrcorrection

def FDR(p_values, fdr, total=None):
    """
    Runs false detection correction for a table of statistics

    Parameters
    ----------
    p_values : ~pandas.DataFrame
        DataFrame with a 'pvalue' column
    fdr : float
        False detection rate
    total : int
        Total number of tests (for multi-enrichment)

    Returns
    -------
    ~pandas.DataFrame
        Table containing entries that passed multiple hypothesis correction
    """

    if total is not None:
        pvals = p_values.p_value.values.tolist() + [1] * (total - len(p_values))
    else:
        pvals = p_values.p_value.values

    keep, qvals = fdrcorrection(pvals, alpha=fdr)

    result = p_values.copy()
    result["qvalue"] = qvals[: len(p_values)]
    result = result[keep[: len(p_values)]]
    return result.sort_values("qvalue")

In [ ]:
FDR(phylon_go_enrichments[phylon_go_enrichments['phylon']==phylon], 0.025, total=None)

In [ ]:
# Explore all phylons:
from pyphylon.biointerp import gen_phylon_wordcloud
for phylon in phylon_go_enrichments['phylon'].unique():
    phylon_enr = phylon_go_enrichments[phylon_go_enrichments['phylon']==phylon]
    phylon_enr.loc[:,'products'] = phylon_enr['products'].str.replace(';', '<br>')
    display(FDR(phylon_enr, 0.025, total=None))